# 第 9 章: Boston データの探索と可視化

標準化の前後で分布がどう変わるか、特徴量と価格がどんな関係にあるかを確認する。

In [ ]:
import sys

sys.path.append("..")

import matplotlib.pyplot as plt
import seaborn as sns
from japanese_font import use_japanese_font

from lib.chapter09.__main__ import COLUMNS
from lib.chapter09.feature_engineering import (
    Standardizer,
    iqr_outliers,
    join_weather,
    load_bike,
    load_weather,
    mean_count_by_weather,
    prepare_boston,
)
from lib.dataset import data_dir

use_japanese_font();

In [ ]:
split = prepare_boston(data_dir() / "Boston.csv", test_size=0.3, seed=0)
train = split.x_train[COLUMNS]
standardized = Standardizer.fit(train).transform(train)
train.shape

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for i, column in enumerate(COLUMNS):
    train[column].plot.hist(ax=axes[0, i], title=f"{column}（標準化前）")
    standardized[column].plot.hist(ax=axes[1, i], title=f"{column}（標準化後）")
fig.tight_layout();

In [ ]:
{
    "標準化前の平均": train.mean().round(2).to_dict(),
    "標準化後の標準偏差": standardized.std(ddof=0).round(2).to_dict(),
}

In [ ]:
priced = train.assign(PRICE=split.t_train)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, column in zip(axes, COLUMNS, strict=True):
    sns.scatterplot(data=priced, x=column, y="PRICE", ax=ax)
fig.tight_layout();

In [ ]:
priced.corr()["PRICE"].round(3)

In [ ]:
split.t_train.plot.box(title="訓練データの PRICE")
int(iqr_outliers(split.t_train).sum())

In [ ]:
joined = join_weather(
    load_bike(data_dir() / "bike.tsv"), load_weather(data_dir() / "weather.csv")
)
mean_count_by_weather(joined).plot.bar(title="天気ごとの平均利用者数");